Load a random repository from the sample and display its feature matrix.

In [ ]:
import sys
import time
from pathlib import Path

project_root = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "src" / "specmine").is_dir()
    ),
    None,
)
if project_root is None:
    raise FileNotFoundError(
        "Could not find src/specmine. Open this notebook from the "
        "FinalProject-CS440 folder or one of its subfolders."
    )
sys.path.insert(0, str(project_root / "src"))
print(f"Notebook Python: {sys.executable}")
print(f"Project source: {project_root / 'src'}")

from specmine.data import load_data

In [ ]:
start_time = time.time()
data = load_data(curated_only=True, sample_size=500, seed=42)
if data is None:
    df = None
    print("Dataset load failed; no data is available.")
else:
    df = data
    print(f"\nDataset loaded successfully in {time.time() - start_time:.2f} seconds")
    print(f"Dataset shape: {df.shape}")
    print(f"Dataset columns: {df.columns.tolist()}")
    print(
        f"Unique repositories: {df['repo_name'].nunique() if 'repo_name' in df else 'missing repo_name'}"
    )
    print("Dataset preview:")
    display(df.head(10))

In [ ]:
import traceback

from specmine.data import load_feature_data
from specmine.features import build_features

if df is None:
    print("No data to display because the dataset load failed.")
elif df.empty:
    print("No data to display because the loaded dataset is empty.")
else:
    repo_names = df["repo_name"].dropna().drop_duplicates()

    if repo_names.empty:
        print("No repositories were found in the loaded dataset.")
    else:
        random_repo = repo_names.sample(n=1).iloc[0]
        repo_df = df.loc[df["repo_name"].eq(random_repo)]
        print(f"Random repository: {random_repo}")
        print(f"Repository spec rows: {len(repo_df)}")
        print(f"Repository IDs: {repo_df['file_url_sha16'].tolist()}")
        print("Repository rows preview:")
        display(repo_df.head(10))
        try:
            feature_load_started = time.time()
            feature_rows = load_feature_data(repo_df)
        except Exception as e:
            print(f"\nError loading feature data: {e}")
            print("Check requirements.txt for needed packages")
            traceback.print_exc()
        else:
            feature_matrix = build_features(feature_rows)

            print(
                f"Feature loading and engineering time: {time.time() - feature_load_started:.2f} seconds"
            )
            print(f"Raw feature rows shape: {feature_rows.shape}")
            print(f"Raw feature columns: {feature_rows.columns.tolist()}")
            print(f"Engineered feature columns: {feature_matrix.columns.tolist()}")
            print(f"Feature matrix shape: {feature_matrix.shape}")
            print("Engineered feature matrix preview:")
            display(feature_matrix.head(20))

In [ ]:
import json
import logging
import os
import traceback

import numpy as np
import pandas as pd

from specmine.evaluate import (
    _attach_labels,
    _expected_columns,
    _load_model,
    _read_labels,
    evaluate_model,
)

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
debug_logger = logging.getLogger("specmine.notebook.evaluate")
model_path = os.environ.get("SPECMINE_MODEL_PATH")
labels_path = os.environ.get("SPECMINE_LABELS_CSV")

print("=== Evaluation debug setup ===")
print(f"Evaluator module: {__import__('specmine.evaluate', fromlist=['']).__file__}")
print(f"Random repository: {random_repo}")
print(f"Rows in repository metadata: {len(repo_df)}")
print(f"Model path from SPECMINE_MODEL_PATH: {model_path!r}")
print(f"Labels path from SPECMINE_LABELS_CSV: {labels_path!r}")

if model_path:
    model = _load_model(Path(model_path))
    print(f"Loaded trained model: {type(model).__module__}.{type(model).__name__}")
    print(f"Expected model feature columns: {_expected_columns(model)}")
else:

    class DebugOnlyModel:
        """Fixed-probability stub to exercise evaluator plumbing, not classification."""

        feature_names_in_ = np.array(["n_words", "n_lines"])
        classes_ = np.array([0, 1])

        def predict_proba(self, features):
            return np.tile([0.5, 0.5], (len(features), 1))

    model = DebugOnlyModel()
    print(
        "WARNING: No trained model configured. Using a fixed 0.5-probability "
        "stub only to smoke-test evaluation; these are not classifier results."
    )

if "feature_matrix" not in globals() or feature_matrix.empty:
    raise RuntimeError("The preceding cell did not produce a feature_matrix.")
if "file_url_sha16" not in repo_df or "file_url_sha16" not in feature_matrix:
    raise KeyError("Both repository rows and features must contain file_url_sha16.")

evaluation_frame = repo_df.merge(
    feature_matrix,
    on="file_url_sha16",
    how="inner",
    validate="one_to_one",
    suffixes=("", "_feature"),
)
expected_columns = _expected_columns(model)
missing_columns = [
    column for column in expected_columns if column not in evaluation_frame
]
print(f"Merged evaluation frame shape: {evaluation_frame.shape}")
print(f"Merged evaluation columns: {evaluation_frame.columns.tolist()}")
print(f"Missing model feature columns: {missing_columns}")
print("Evaluation frame dtypes:")
print(evaluation_frame.dtypes.to_string())
print("Evaluation frame preview:")
display(evaluation_frame.head(10))
if missing_columns:
    raise ValueError(f"Evaluation frame is missing model columns: {missing_columns}")

if labels_path:
    if not model_path:
        raise ValueError("Set SPECMINE_MODEL_PATH before scoring external labels.")
    labels = _read_labels(Path(labels_path))
    print(
        f"Labels loaded: {len(labels)} rows; label counts:\n{labels['label'].value_counts(dropna=False)}"
    )
    evaluation_frame = _attach_labels(evaluation_frame, labels)
    n_unmatched = int((~evaluation_frame["_label_matched"]).sum())
    n_unlabeled = int(
        (evaluation_frame["_label_matched"] & evaluation_frame["label"].isna()).sum()
    )
    print(f"Unmatched rows: {n_unmatched}; matched rows missing labels: {n_unlabeled}")
    evaluation_frame = evaluation_frame.loc[evaluation_frame["label"].notna()].drop(
        columns="_label_matched"
    )
    if evaluation_frame.empty:
        raise ValueError(
            "No labeled rows from the random repository matched the labels CSV."
        )
else:
    print("No label CSV configured; supervised metrics will be skipped.")

print("=== Running evaluate_model ===")
try:
    evaluation_result = evaluate_model(model, evaluation_frame, threshold=0.5)
except Exception:
    debug_logger.exception("Evaluation failed")
    traceback.print_exc()
    raise

print("Evaluation result (full JSON):")
print(json.dumps(evaluation_result, indent=2, default=str))
print("Per-example predictions:")
display(pd.DataFrame(evaluation_result["predictions"]))
if evaluation_result["metrics"] is None:
    print(
        "No ground truth was supplied: no accuracy or calibration metrics were computed."
    )
else:
    print("Metrics:")
    print(json.dumps(evaluation_result["metrics"], indent=2, default=str))